# Adj Normal, Tumor, and CAR macrophages

Reproduced from `Sx_DR18vsCAR.ipynb`. Run this notebook with an R/IRkernel using Seurat.

The object-preparation cell loads `combined_macrophages_normalized.rds` when available. On the first run, or if that file is lost, it rebuilds the object from the two original input files and saves the RDS after normalization. All rebuilding source code is retained below. Set `force_rebuild <- TRUE` to refresh the cache after changing input files or preprocessing; set it back to `FALSE` for subsequent runs. Run from this notebook's folder.


In [ ]:
# Avoid LaTeX list output unsupported by the notebook viewer.
options(jupyter.display_mimetypes = setdiff(getOption("jupyter.display_mimetypes"), "text/latex"))
library(Seurat)
library(patchwork)
library(ggplot2)

In [ ]:
# Cache the finished object; set TRUE after changing inputs or preprocessing.
cache_file <- "combined_macrophages_normalized.rds"
force_rebuild <- FALSE

if (file.exists(cache_file) && !force_rebuild) {
  message("Loading cached object: ", cache_file)
  combined <- readRDS(cache_file)
  stopifnot(inherits(combined, "Seurat"),
            "RNA" %in% names(combined@assays),
            "condition" %in% names(combined@meta.data))
} else {
  message("Building combined object from source files...")
  # CAR macrophages: combine the AAVS1 and SIRPa samples under one label.
  sx2 <- readRDS("three_sample_macrophage_CAR_renamed.rds")
  sx_carm <- subset(sx2, subset = orig.ident == "CAR")
  sx_carm$condition <- "CAR_M"

  # HCC macrophages: retain only adjacent-normal and tumour samples.
  hcc <- readRDS("HCCF1F4.RData")
  hcc <- UpdateSeuratObject(hcc)
  Idents(hcc) <- "CellType"
  sharma_myeloid <- subset(hcc, idents = "Myeloid")
  Idents(sharma_myeloid) <- "condition"
  sharma_myeloidsub <- subset(sharma_myeloid, idents = c("Adj Normal", "Tumor"))
  # One object containing all three comparison groups.
  combined <- merge(sharma_myeloidsub, sx_carm)
  combined$condition <- factor(
    combined$condition,
    levels = c("CAR_M", "Tumor", "Adj Normal")
  )

  # Required by Seurat v5 if the RNA assay still has split layers.
  combined <- JoinLayers(combined, assay = "RNA")
  combined <- NormalizeData(combined, assay = "RNA")
  DefaultAssay(combined) <- "RNA"
  table(combined$condition)

  # Save only after all preprocessing succeeds. A temporary file avoids
  # leaving a partial cache if saving is interrupted.
  temporary_cache <- tempfile(pattern = ".combined_", tmpdir = dirname(cache_file))
  tryCatch({
    saveRDS(combined, temporary_cache)
    if (!file.rename(temporary_cache, cache_file)) {
      stop("Could not move the completed RDS to: ", cache_file)
    }
  }, finally = {
    if (file.exists(temporary_cache)) unlink(temporary_cache)
  })
  message("Saved cached object: ", normalizePath(cache_file))
}


In [ ]:
# Available after either loading the cache or rebuilding it.
table(combined$condition)


In [ ]:
# Prepare the requested 20-marker panel in the same order as supplied.
# Human marker aliases are mapped to their RNA feature symbols for lookup.
requested_gene_map <- c(
  IL18BP = "IL18BP", CD274 = "CD274", LRIG2 = "LRIG2", MARCO = "MARCO",
  IFITM1 = "IFITM1", IFITM2 = "IFITM2", IFIT2 = "IFIT2", IRF1 = "IRF1",
  CD273 = "PDCD1LG2", CD206 = "MRC1", ARG = "ARG1", iNOS = "NOS2",
  TGFB1 = "TGFB1", IL10 = "IL10", IL4 = "IL4", IL13 = "IL13",
  TGFB3 = "TGFB3", SPP1 = "SPP1", TREM2 = "TREM2", VSIG4 = "VSIG4"
)
rna_features <- rownames(combined[["RNA"]])
feature_index <- match(toupper(unname(requested_gene_map)), toupper(rna_features))
missing_genes <- names(requested_gene_map)[is.na(feature_index)]
genes_req <- rna_features[feature_index[!is.na(feature_index)]]
names(genes_req) <- names(requested_gene_map)[!is.na(feature_index)]
if (!length(genes_req)) stop("None of the requested genes are present in the RNA assay.")
if (length(missing_genes)) message("Genes not found: ", paste(missing_genes, collapse = ", "))
gene_labels <- setNames(names(genes_req), unname(genes_req))

# Read normalized RNA expression directly for Seurat v4/v5 compatibility.
expression_matrix <- if ("LayerData" %in% getNamespaceExports("SeuratObject")) {
  SeuratObject::LayerData(combined, assay = "RNA", layer = "data")[unname(genes_req), , drop = FALSE]
} else {
  SeuratObject::GetAssayData(combined, assay = "RNA", slot = "data")[unname(genes_req), , drop = FALSE]
}


In [ ]:
# Optional bar-chart view of the same normalized values shown in the violin plots.
# Bars show the mean across pooled cells; error bars show SD across cells.
# These are descriptive cell-level summaries, not biological-replicate estimates.
bar_conditions <- c("CAR_M", "Tumor", "Adj Normal")
bar_cell_conditions <- as.character(combined$condition[colnames(expression_matrix)])

bar_summary <- do.call(rbind, lapply(unname(genes_req), function(gene) {
  do.call(rbind, lapply(bar_conditions, function(condition) {
    values <- as.numeric(expression_matrix[gene, bar_cell_conditions == condition])
    data.frame(
      gene = unname(gene_labels[gene]),
      condition = condition,
      mean_expression = mean(values),
      sd_cells = if (length(values) > 1L) sd(values) else NA_real_,
      cells = length(values)
    )
  }))
}))
bar_summary$gene <- factor(bar_summary$gene, levels = unname(gene_labels[unname(genes_req)]))
bar_summary$condition <- factor(bar_summary$condition, levels = bar_conditions)

bar_colors <- c(CAR_M = "#1F6F5F", Tumor = "#6F6F6F", `Adj Normal` = "#B6B6B0")

ggplot(bar_summary, aes(x = condition, y = mean_expression, fill = condition)) +
  geom_col(width = 0.68) +
  geom_errorbar(
    aes(ymin = pmax(0, mean_expression - sd_cells),
        ymax = mean_expression + sd_cells),
    width = 0.18, linewidth = 0.35, na.rm = TRUE
  ) +
  facet_wrap(~gene, scales = "free_y", ncol = 4) +
  scale_fill_manual(values = bar_colors, drop = FALSE) +
  labs(
    title = "Mean normalized expression by condition",
    x = NULL,
    y = "Mean RNA expression (± SD across cells)"
  ) +
  theme_classic() +
  theme(
    legend.position = "none",
    strip.background = element_blank(),
    strip.text = element_text(face = "bold"),
    axis.text.x = element_text(angle = 35, hjust = 1),
    panel.spacing = grid::unit(1, "lines")
  )


In [ ]:
options(repr.plot.width = 16, repr.plot.height = 12)

# Bar chart of the requested genes by condition.
# Error bars show SD across pooled cells; summaries are descriptive, not biological-replicate estimates.
bar_conditions <- c("CAR_M", "Tumor", "Adj Normal")
bar_cell_conditions <- as.character(combined$condition[colnames(expression_matrix)])

bar_summary <- do.call(rbind, lapply(unname(genes_req), function(gene) {
  do.call(rbind, lapply(bar_conditions, function(condition) {
    values <- as.numeric(expression_matrix[gene, bar_cell_conditions == condition])
    data.frame(
      gene = unname(gene_labels[gene]),
      condition = condition,
      mean_expression = mean(values),
      sd_cells = if (length(values) > 1L) sd(values) else NA_real_,
      cells = length(values)
    )
  }))
}))
bar_summary$gene <- factor(bar_summary$gene, levels = unname(gene_labels[unname(genes_req)]))
bar_summary$condition <- factor(bar_summary$condition, levels = bar_conditions)

bar_colors <- c(CAR_M = "#1F6F5F", Tumor = "#6F6F6F", `Adj Normal` = "#B6B6B0")

ggplot(bar_summary, aes(x = condition, y = mean_expression, fill = condition)) +
  geom_col(width = 0.68) +
  geom_errorbar(
    aes(ymin = pmax(0, mean_expression - sd_cells),
        ymax = mean_expression + sd_cells),
    width = 0.18, linewidth = 0.35, na.rm = TRUE
  ) +
  facet_wrap(~gene, scales = "free_y", ncol = 4) +
  scale_fill_manual(values = bar_colors, drop = FALSE) +
  labs(
    title = "Mean normalized expression by condition",
    subtitle = "Cell-level means; SD across cells; y-axis scales vary by marker",
    x = NULL,
    y = "Mean normalized RNA expression"
  ) +
  theme_classic(base_size = 10) +
  theme(
    legend.position = "none",
    strip.background = element_blank(),
    strip.text = element_text(face = "bold"),
    axis.text.x = element_text(angle = 35, hjust = 1),
    panel.spacing = grid::unit(1, "lines")
  )

conditions <- c("CAR_M", "Tumor", "Adj Normal")
cell_conditions <- as.character(combined$condition[colnames(expression_matrix)])

expression_columns <- unlist(lapply(unname(genes_req), function(gene) {
  lapply(conditions, function(condition) {
    as.numeric(expression_matrix[gene, cell_conditions == condition])
  })
}), recursive = FALSE)

names(expression_columns) <- unlist(lapply(unname(genes_req), function(gene) {
  paste(unname(gene_labels[gene]), conditions, sep = "_")
}))

# Pad shorter groups with blank cells so every gene/group is one CSV column.
max_cells <- max(lengths(expression_columns))
expression_export <- as.data.frame(lapply(expression_columns, function(values) {
  c(values, rep(NA_real_, max_cells - length(values)))
}))

output_file <- "RNA_expression_by_gene_and_condition.csv"
write.csv(expression_export, output_file, row.names = FALSE, na = "")
message("Exported: ", normalizePath(output_file))


In [ ]:
# Export requested markers in the same wide gene-by-condition format as RNA_expression_by_gene_and_condition.csv.
marker_aliases <- c(
  IL18BP = "IL18BP",
  CD273 = "PDCD1LG2",
  LRIG2 = "LRIG2",
  MARCO = "MARCO",
  IFITM1 = "IFITM1",
  IFITM2 = "IFITM2",
  IFIT2 = "IFIT2",
  IRF1 = "IRF1",
  CD206 = "MRC1",
  ARG1 = "ARG1",
  iNOS = "NOS2",
  TGFB1 = "TGFB1",
  IL10 = "IL10",
  IL4 = "IL4",
  IL13 = "IL13",
  TGFB3 = "TGFB3",
  SPP1 = "SPP1",
  TREM2 = "TREM2",
  VSIG4 = "VSIG4"
)
conditions <- c("CAR_M", "Tumor", "Adj Normal")

rna_features <- rownames(combined[["RNA"]])
export_genes <- marker_aliases[marker_aliases %in% rna_features]
missing_markers <- names(marker_aliases)[!marker_aliases %in% rna_features]
if (!length(export_genes)) stop("None of the requested markers are present in the RNA assay.")

normalized_expression <- if ("LayerData" %in% getNamespaceExports("SeuratObject")) {
  SeuratObject::LayerData(combined, assay = "RNA", layer = "data")[export_genes, , drop = FALSE]
} else {
  SeuratObject::GetAssayData(combined, assay = "RNA", slot = "data")[export_genes, , drop = FALSE]
}
cell_conditions <- as.character(combined$condition[colnames(normalized_expression)])

expression_columns <- unlist(lapply(names(export_genes), function(marker) {
  lapply(conditions, function(condition) {
    as.numeric(normalized_expression[export_genes[[marker]], cell_conditions == condition])
  })
}), recursive = FALSE)
names(expression_columns) <- unlist(lapply(names(export_genes), function(marker) {
  paste(marker, conditions, sep = "_")
}))

max_cells <- max(lengths(expression_columns))
expression_export <- as.data.frame(lapply(expression_columns, function(values) {
  c(values, rep(NA_real_, max_cells - length(values)))
}))
names(expression_export) <- gsub("Adj Normal", "Adj.Normal", names(expression_export), fixed = TRUE)

xml_escape <- function(value) {
  value <- gsub("&", "&amp;", value, fixed = TRUE)
  value <- gsub("<", "&lt;", value, fixed = TRUE)
  value <- gsub(">", "&gt;", value, fixed = TRUE)
  value <- gsub("\"", "&quot;", value, fixed = TRUE)
  gsub("'", "&apos;", value, fixed = TRUE)
}

excel_column <- function(index) {
  result <- ""
  while (index > 0) {
    remainder <- (index - 1) %% 26
    result <- paste0(LETTERS[remainder + 1], result)
    index <- (index - 1) %/% 26
  }
  result
}

xlsx_cell <- function(value, cell_ref) {
  if (is.numeric(value) && !is.na(value)) {
    paste0('<c r="', cell_ref, '"><v>', format(value, scientific = FALSE, trim = TRUE), "</v></c>")
  } else {
    paste0('<c r="', cell_ref, '" t="inlineStr"><is><t>', xml_escape(ifelse(is.na(value), "", as.character(value))), "</t></is></c>")
  }
}

worksheet_rows <- vapply(seq_len(nrow(expression_export) + 1), function(row_number) {
  values <- if (row_number == 1) names(expression_export) else expression_export[row_number - 1, , drop = FALSE]
  cells <- vapply(seq_along(values), function(column_number) {
    xlsx_cell(values[[column_number]], paste0(excel_column(column_number), row_number))
  }, character(1))
  paste0('<row r="', row_number, '">', paste0(cells, collapse = ""), "</row>")
}, character(1))

xlsx_dir <- tempfile("xlsx_")
dir.create(file.path(xlsx_dir, "_rels"), recursive = TRUE)
dir.create(file.path(xlsx_dir, "xl", "_rels"), recursive = TRUE)
dir.create(file.path(xlsx_dir, "xl", "worksheets"), recursive = TRUE)

writeLines(c(
  '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>',
  '<Types xmlns="http://schemas.openxmlformats.org/package/2006/content-types">',
  '<Default Extension="rels" ContentType="application/vnd.openxmlformats-package.relationships+xml"/>',
  '<Default Extension="xml" ContentType="application/xml"/>',
  '<Override PartName="/xl/workbook.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet.main+xml"/>',
  '<Override PartName="/xl/worksheets/sheet1.xml" ContentType="application/vnd.openxmlformats-officedocument.spreadsheetml.worksheet+xml"/>',
  "</Types>"
) , file.path(xlsx_dir, "[Content_Types].xml"))
writeLines(c(
  '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>',
  '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">',
  '<Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/officeDocument" Target="xl/workbook.xml"/>',
  "</Relationships>"
) , file.path(xlsx_dir, "_rels", ".rels"))
writeLines(c(
  '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>',
  '<workbook xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main" xmlns:r="http://schemas.openxmlformats.org/officeDocument/2006/relationships">',
  '<sheets><sheet name="Expression" sheetId="1" r:id="rId1"/></sheets>',
  "</workbook>"
) , file.path(xlsx_dir, "xl", "workbook.xml"))
writeLines(c(
  '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>',
  '<Relationships xmlns="http://schemas.openxmlformats.org/package/2006/relationships">',
  '<Relationship Id="rId1" Type="http://schemas.openxmlformats.org/officeDocument/2006/relationships/worksheet" Target="worksheets/sheet1.xml"/>',
  "</Relationships>"
) , file.path(xlsx_dir, "xl", "_rels", "workbook.xml.rels"))
writeLines(c(
  '<?xml version="1.0" encoding="UTF-8" standalone="yes"?>',
  '<worksheet xmlns="http://schemas.openxmlformats.org/spreadsheetml/2006/main"><sheetData>',
  worksheet_rows,
  "</sheetData></worksheet>"
) , file.path(xlsx_dir, "xl", "worksheets", "sheet1.xml"))

if (length(missing_markers)) message("Markers not found: ", paste(missing_markers, collapse = ", "))

In [ ]:
# Write the generated workbook into this project's Mac_analysis folder.
output_file <- "requested_markers_expression.xlsx"
previous_directory <- getwd()
setwd(xlsx_dir)
utils::zip(output_file, list.files(".", recursive = TRUE, all.files = TRUE, no.. = TRUE))
setwd(previous_directory)
message("Exported: ", output_file)